# NAMAA Kaggle V3 — isolated, version-pinned GPU environment
Based on official Chatterbox 0.1.7 requirements and PyTorch CUDA 12.6 wheels. No Project 79 server. Run cells top to bottom. Produces namaa_test.wav only after successful generation.

In [ ]:
import os, sys, subprocess
from pathlib import Path
venv=Path("/kaggle/working/namaa_venv_v3")
py=venv/"bin/python"
env=os.environ.copy()
env["PIP_NO_CACHE_DIR"]="1"
env["HF_HOME"]="/kaggle/working/namaa_hf_cache"
def step(msg,cmd):
    print("STEP",msg,flush=True)
    subprocess.run(cmd,check=True,env=env)
if not py.exists():
    step("CREATE_ISOLATED_VENV",[sys.executable,"-m","venv",str(venv)])
step("PIP_TOOLING",[str(py),"-m","pip","install","--disable-pip-version-check","-q","--upgrade","pip"])
step("PIN_MATCHED_CUDA_WHEELS",[str(py),"-m","pip","install","--disable-pip-version-check","-q","torch==2.6.0","torchaudio==2.6.0","--index-url","https://download.pytorch.org/whl/cu126"])
step("INSTALL_OFFICIAL_CHATTERBOX",[str(py),"-m","pip","install","--disable-pip-version-check","-q","chatterbox-tts==0.1.7","huggingface_hub"])
step("PREFLIGHT_FRESH_PROCESS",[str(py),"-c","import torch,torchaudio;from chatterbox import mtl_tts;print('PRECHECK_OK',torch.__version__,torchaudio.__version__,torch.cuda.is_available(),flush=True); assert torch.__version__.split('+')[0]=='2.6.0';assert torchaudio.__version__.split('+')[0]=='2.6.0';assert torch.cuda.is_available()"])
print("ENVIRONMENT_READY",str(py),flush=True)


In [ ]:
import os,subprocess
from pathlib import Path
script=Path("/kaggle/working/namaa_infer_v3.py")
script.write_text("import os, sys, time\nfrom pathlib import Path\nimport torch\nimport torchaudio\nfrom huggingface_hub import snapshot_download\nfrom safetensors.torch import load_file as load_safetensors\nfrom chatterbox import mtl_tts\n\ndef log(*xs):\n    print(*xs, flush=True)\n\nlog(\"PYTHON\", sys.version.split()[0], \"TORCH\", torch.__version__, \"TORCHAUDIO\", torchaudio.__version__)\nassert torch.__version__.split(\"+\")[0]==\"2.6.0\", f\"Unexpected torch {torch.__version__}\"\nassert torchaudio.__version__.split(\"+\")[0]==\"2.6.0\", f\"Unexpected torchaudio {torchaudio.__version__}\"\nassert torch.cuda.is_available(), \"GPU T4 unavailable\"\nlog(\"GPU\", torch.cuda.get_device_name(0))\nstart=time.monotonic()\nlog(\"BASE_MODEL_LOADING\")\nmodel=mtl_tts.ChatterboxMultilingualTTS.from_pretrained(device=\"cuda\")\nlog(\"BASE_MODEL_READY\",round(time.monotonic()-start,1))\nlog(\"NAMAA_WEIGHT_LOADING\")\nfolder=snapshot_download(repo_id=\"NAMAA-Space/NAMAA-Egyptian-TTS\",repo_type=\"model\",allow_patterns=[\"t3_mtl23ls_v2.safetensors\"])\nstate=load_safetensors(str(Path(folder)/\"t3_mtl23ls_v2.safetensors\"),device=\"cpu\")\nmodel.t3.load_state_dict(state,strict=True)\nmodel.t3.to(\"cuda\").eval()\nlog(\"NAMAA_READY\",round(time.monotonic()-start,1))\ntext=\"إزيك يا جماعة، بنجرّب الصوت المصري. النهارده هنحكي حكاية مانسا موسى.\"\nlog(\"GENERATING\",text)\nt=time.monotonic()\nwith torch.inference_mode():\n    wav=model.generate(text,language_id=\"ar\")\nout=Path(\"/kaggle/working/namaa_test.wav\")\ntorchaudio.save(str(out),wav.detach().cpu(),model.sr)\nseconds=round(wav.shape[-1]/model.sr,2)\nlog(\"SUCCESS\",str(out),\"BYTES\",out.stat().st_size,\"AUDIO_SECONDS\",seconds,\"GENERATE_SECONDS\",round(time.monotonic()-t,2),\"TOTAL_SECONDS\",round(time.monotonic()-start,1))\n",encoding="utf-8")
env=os.environ.copy()
env["HF_HOME"]="/kaggle/working/namaa_hf_cache"
print("START_IN_FRESH_PYTHON_PROCESS",flush=True)
subprocess.run(["/kaggle/working/namaa_venv_v3/bin/python","-u",str(script)],env=env,check=True)
